# 📊 11｜業務週報：比率要先講清楚分子與分母

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/PythonForDataAnalysis/blob/master/AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90_12%E5%80%8B%E7%AF%84%E4%BE%8B/11_sales_kpi_style/lesson.ipynb)

> 💡 **學習模式說明**：
> 本筆記本專為**完全不寫程式的初學者**設計，全程搭配 **Google Colab 內建的 Gemini AI** 進行協作。
> 你不需要手寫程式碼：先用 Prompt 指揮 AI，再對照下方已附的參考程式，執行並驗收數字與圖表。
> 🆕 **本例只新增一個重點**：「達成率」是比率：整季達成率＝**整季實績加總 ÷ 整季目標加總**，不是把 13 週的達成率拿來平均。報表數字要加上千分位與百分比格式，讓人一眼看懂。

## 🪜 本例路線：先完成一次，再逐步加深
1. **必做主線（單元一～三）**：寫出達成率的分子與分母，比較兩種算法，最後用**一張垂直長條圖**回答「哪一組的整季達成率最高？」。單元三後先停下來驗收。
2. **報表應用（單元四）**：匯出有千分位與百分比格式的 Excel 週報（組別、個人兩個工作表）。
3. **選做挑戰（單元五～六）**：找出未達標的業務，或觀察 13 週實績與季末衝刺。

---

## 🧭 資料專案介紹：貫徹「看懂資料與選對圖表」思維 SOP

資料：`data/sales_kpi.csv`（請先閱讀[本例資料卡](./data/DATA_CARD.md)）。**一列代表：一位業務、一週的目標與實績**，15 位業務 × 13 週＝195 列（2025 年第一季）。

### 1. 欄位角色貼標籤
- ⚪ **標識 (ID)**：`業務` ➔ 只用來分組或辨識。
- 🔵 **條件 (Dimension)**：`組別`（類別）、`週次`（時間，1～13）。
- 🟢 **指標 (Metric)**：`週目標`（元，分母）、`週實績`（元，分子）、`新客數`（人）、`退貨筆數`（筆）。
- 🟢 **衍生比率**：`達成率`＝實績 ÷ 目標 ➔ 整季達成率＝**整季實績合計 ÷ 整季目標合計**。

### 2. 五大提問積木與核心問題
- 🎯 **比較（必做）**：哪一組的整季達成率最高？三組都達標（≥100%）了嗎？
- 👤 **比較（選做）**：15 位業務中，誰的整季達成率未達 100%？
- 📈 **趨勢（選做）**：全公司 13 週的實績如何變化？季末兩週是否特別高？

### 3. 邊界反向檢核防呆（不可觸犯的紅線）
- ⚠️ **比率先寫分子與分母**：達成率＝實績 ÷ 目標；整季達成率是「整季實績合計 ÷ 整季目標合計」。
- ⚠️ **不要平均比率**：把 13 週的達成率直接平均，等於假設每週、每人的目標一樣重要；目標不同時答案會偏掉。
- ⚠️ **格式只改顯示，不改數值**：千分位、百分比是給人看的；Excel 儲存格仍要保留可計算的數值。
- ⚠️ **季末衝刺要說明**：第 12、13 週業績特別高，是季末衝業績，不代表下一季也會持續成長。
- ⚠️ **達成率不是能力**：目標高低不同（18～22 萬），達成率高不一定代表業績金額最多，評價個人時要一起看。

### 4. 決定最適圖表
- **3 組比整季達成率** ➔ **垂直長條圖**（Y 軸從 0 開始，單位 %；加上 100% 目標參考線）。
- **15 位業務達成率（選做）** ➔ **水平長條圖**（人數多、依達成率排序；未達 100% 用醒目顏色）。
- **13 週實績與目標（選做）** ➔ **折線圖**（實績與目標兩條線，單位相同，可以共用 Y 軸）。

---

## 🤖 Colab Gemini AI 操作方式
1. 每次只複製一個單元的 Prompt，貼到 Colab Gemini 對話框。
2. 比較 AI 的答覆與下方參考程式，執行目前單元的儲存格。
3. 先看懂輸出並回答驗收問題，再進入下一單元。

In [ ]:
# 第 0 步：下載範例資料並安裝繁體中文字型（請先點擊左側 ▶ 執行此儲存格）
import os
from pathlib import Path
from urllib.request import urlretrieve

# 1. 下載資料（已存在就不重複下載；若下載失敗，請手動上傳到 data/）
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
files = {
    'sales_kpi.csv': 'https://raw.githubusercontent.com/roberthsu2003/PythonForDataAnalysis/master/AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90_12%E5%80%8B%E7%AF%84%E4%BE%8B/11_sales_kpi_style/data/sales_kpi.csv',
}
for name, url in files.items():
    if not (data_dir / name).exists():
        urlretrieve(url, data_dir / name)
print('📂 data/ 內的檔案：', sorted(p.name for p in data_dir.iterdir()))

# 2. 安裝與設定 Colab 繁體中文字型 (思源黑體)
%pip install -q mplfonts
import glob
import mplfonts
from matplotlib import font_manager
from mplfonts import use_font

for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
    font_manager.fontManager.addfont(f)
use_font("Noto Sans CJK SC")

print("✅ 環境初始化完成！資料已就緒，繁體中文字型已設定成功。請開始進行下方單元。")

### 📝 單元一【必做】：讀取資料，找出比率的分子與分母（貼標籤）

> 🎯 **思維重點**：原始資料沒有「達成率」欄；先確認一列代表什麼，並找出達成率的分子（週實績）與分母（週目標）。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請幫我用 pandas 讀取 data/sales_kpi.csv：
1. 顯示前 5 筆資料、所有欄位名稱、總列數與各欄缺值數。
2. 計算有幾位業務、幾個組別、幾週，確認「業務數 × 週數」是否等於總列數。
3. 列出每位業務的週目標（是否每週固定？）。
4. 用文字說明：一列代表什麼？哪些欄位是標識、條件、指標？「達成率」的分子與分母分別是哪一欄？
先不要畫圖。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
import pandas as pd

df = pd.read_csv('data/sales_kpi.csv')
print("--- 前 5 筆資料 ---")
display(df.head())
print(f"\n總列數：{len(df)}；缺值：{df.isna().sum().sum()}")
print(f"業務 {df['業務'].nunique()} 位 × 週數 {df['週次'].nunique()} 週 = {df['業務'].nunique() * df['週次'].nunique()} 列")
print("各組人數：", df.groupby('組別')['業務'].nunique().to_dict())
print("每位業務的週目標是否固定：", (df.groupby('業務')['週目標'].nunique() == 1).all())
print("週目標種類：", sorted(df['週目標'].unique()))
print("\n➡️ 一列代表：一位業務、一週的目標與實績；達成率 = 週實績（分子）÷ 週目標（分母）")

### 📝 單元二【必做】：邊界檢核：兩種達成率算法比一比，並設定表格格式

> 🎯 **思維重點**：「先加總再相除」才是整季達成率；「把每週達成率平均」是常見錯誤。用格式化表格呈現：金額加千分位，比率顯示成百分比。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請依組別計算整季達成率，並比較兩種算法：
1. 正確算法：整季實績合計 ÷ 整季目標合計。
2. 常見錯誤：先算每一列的週達成率，再對組別取平均。
3. 整理成一張表：組別、整季目標、整季實績、整季達成率（正確）、平均週達成率（錯誤示範），依正確達成率由高到低排序。
4. 用 pandas Styler 設定顯示格式：金額加千分位（例如 12,353,000），比率顯示為百分比、保留一位小數。
5. 用文字說明：兩種算法為什麼會不一樣？
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
group_df = df.groupby('組別')[['週目標', '週實績']].sum().rename(columns={'週目標': '整季目標', '週實績': '整季實績'})
group_df['整季達成率'] = group_df['整季實績'] / group_df['整季目標']            # ✅ 先加總、再相除
group_df['平均週達成率（錯誤示範）'] = (df['週實績'] / df['週目標']).groupby(df['組別']).mean()  # ❌ 平均比率
group_df = group_df.sort_values('整季達成率', ascending=False)

display(group_df.style.format({
    '整季目標': '{:,.0f}', '整季實績': '{:,.0f}',
    '整季達成率': '{:.1%}', '平均週達成率（錯誤示範）': '{:.1%}',
}))
print("💡 平均週達成率讓每一週、每一個人的權重都一樣；但目標金額不同（18～22 萬），")
print("   目標高的人達成率應該占比較大的份量，所以要「先加總、再相除」。")

### 📝 單元三【必做】：提問積木【比較】— 哪一組的整季達成率最高？（垂直長條圖）

> 🎯 **思維重點**：3 個組別比高低，用【垂直長條圖】，Y 軸從 0 開始；加上 100% 目標參考線，柱子上標百分比。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請用正確的整季達成率畫出 3 組的垂直長條圖：
1. X 軸為組別，Y 軸為整季達成率（%），從 0 開始。
2. 加上一條 100% 的水平虛線，標示為「目標 100%」。
3. 柱子上方標註達成率（百分比，一位小數）；達成率最高的組別用醒目顏色。
4. 標題註明「整季實績 ÷ 整季目標」，圖表文字全部使用繁體中文。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
import matplotlib.pyplot as plt

rate = group_df['整季達成率'] * 100
top_group = rate.idxmax()
colors = ['#e74c3c' if g == top_group else '#3498db' for g in rate.index]

plt.figure(figsize=(8, 5))
bars = plt.bar(rate.index, rate.values, color=colors, width=0.5)
for bar in bars:
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2, f'{bar.get_height():.1f}%', ha='center', fontweight='bold')
plt.axhline(100, color='gray', linestyle='--', linewidth=1.5, label='目標 100%')
plt.ylim(0, rate.max() * 1.15)
plt.title(f'各組第一季整季達成率（整季實績 ÷ 整季目標；最高：{top_group}）', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('組別')
plt.ylabel('整季達成率（%）')
plt.legend(loc='lower right')
plt.show()

## ✅ 必做主線驗收：現在先停下來

✏️ **先不要看答案！** 對照上面的表格和圖，自己回答下列問題，把你的答案寫在紙上或筆記裡：

- [ ] 能說出達成率的分子（實績）與分母（目標），並說明整季達成率為什麼要「先加總再相除」。
- [ ] 能展示兩種算法的結果，並說明用哪一個；表格金額有千分位、比率有百分比格式。
- [ ] 能從長條圖與表格指出整季達成率最高的組別，並核對數字一致；圖上有 100% 參考線。
- 💬 **討論題**：北區達成率最高，是否代表北區每一位業務都表現最好？

<details>
<summary>👉 寫好了嗎？點此展開參考答案核對</summary>

- 資料 195 列＝15 位業務 × 13 週；整季目標合計 39,260,000 元。
- 正確算法（先加總再相除）：**北區 116.0%**、南區 103.7%、中區 101.1%；平均每週達成率則是 115.6%、104.1%、101.0%——數字不同。
- 長條圖 Y 軸從 0 開始，有 100% 參考線；三組都達標，北區最高。

</details>

若今天繼續做報表，請進行**單元四**；單元五、六是選做挑戰。

---

### 📝 單元四【報表應用】：匯出有千分位與百分比格式的 Excel 週報

> 🎯 **思維重點**：Excel 報表的數字要「看起來」有格式，但儲存格內仍是可計算的數值；用 openpyxl 設定儲存格的數字格式。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請將達成率分析匯出成格式化的 Excel：
1. 工作表「組別達成率」：組別、整季目標、整季實績、整季達成率。
2. 工作表「個人達成率」：組別、業務、整季目標、整季實績、整季達成率，依達成率由高到低排序。
3. 用 openpyxl 設定格式：金額欄 #,##0、達成率欄 0.0%；標題列深藍底白字、凍結第一列；達成率未達 100% 的儲存格用淺紅底。
4. 存為 output/第一季業務達成率.xlsx（若 output 目錄不存在請自動建立）；重新讀取確認達成率仍是數值（例如 1.16），不是文字。
5. 提供 Colab 下載該 Excel 檔案的程式碼。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill

person_df = df.groupby(['組別', '業務'])[['週目標', '週實績']].sum().rename(columns={'週目標': '整季目標', '週實績': '整季實績'})
person_df['整季達成率'] = person_df['整季實績'] / person_df['整季目標']
person_df = person_df.sort_values('整季達成率', ascending=False).reset_index()

os.makedirs('output', exist_ok=True)
output_path = 'output/第一季業務達成率.xlsx'
with pd.ExcelWriter(output_path) as writer:
    group_df[['整季目標', '整季實績', '整季達成率']].reset_index().to_excel(writer, sheet_name='組別達成率', index=False)
    person_df.to_excel(writer, sheet_name='個人達成率', index=False)

wb = load_workbook(output_path)
for ws in wb.worksheets:
    ws.freeze_panes = 'A2'
    headers = {cell.value: cell.column_letter for cell in ws[1]}
    for cell in ws[1]:
        cell.fill = PatternFill('solid', start_color='1F497D')
        cell.font = Font(bold=True, color='FFFFFF')
    for col in ['整季目標', '整季實績']:
        for cell in ws[headers[col]][1:]:
            cell.number_format = '#,##0'
    for cell in ws[headers['整季達成率']][1:]:
        cell.number_format = '0.0%'
        if cell.value < 1:
            cell.fill = PatternFill('solid', start_color='FCE4D6')
    for letter in headers.values():
        ws.column_dimensions[letter].width = 14
wb.save(output_path)

check = pd.read_excel(output_path, sheet_name='組別達成率')
print("重新讀取的達成率（應為數值）：", check['整季達成率'].round(3).tolist())

try:
    from google.colab import files
    files.download(output_path)
    print("✅ 已啟動瀏覽器下載 Excel 報表！")
except Exception:
    print(f"✅ Excel 檔案已儲存於：{output_path}")

### 📝 單元五【選做挑戰】：提問積木【比較】— 哪些業務未達標？（水平長條圖）

> 🎯 **思維重點**：15 個人比較，名字多，用【水平長條圖】依達成率排序；未達 100% 用醒目顏色。達成率高不等於業績金額最多。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請用單元四的個人整季達成率畫水平長條圖：
1. Y 軸為「業務（組別）」，X 軸為整季達成率（%），從 0 開始，依達成率排序（最高在上）。
2. 加上 100% 垂直參考線；未達 100% 的業務用紅色，其他用藍色；長條右側標註百分比。
3. 列出未達標業務的姓名、組別、達成率與整季實績金額。
4. 用文字提醒：達成率最高的人，實績金額不一定最多；評價時要同時看目標與金額。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
person_plot = person_df.sort_values('整季達成率')
labels = person_plot['業務'] + '（' + person_plot['組別'] + '）'
rates = person_plot['整季達成率'] * 100
colors = ['#e74c3c' if r < 100 else '#3498db' for r in rates]

plt.figure(figsize=(9, 7))
bars = plt.barh(labels, rates, color=colors)
for bar in bars:
    plt.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2, f'{bar.get_width():.1f}%', va='center', fontsize=9)
plt.axvline(100, color='gray', linestyle='--', linewidth=1.5)
plt.xlim(0, rates.max() * 1.12)
plt.title('個人整季達成率（紅色＝未達 100%）', fontsize=13, fontweight='bold')
plt.xlabel('整季達成率（%）')
plt.show()

below = person_df[person_df['整季達成率'] < 1]
display(below.style.format({'整季目標': '{:,.0f}', '整季實績': '{:,.0f}', '整季達成率': '{:.1%}'}))

### 📝 單元六【選做挑戰】：提問積木【趨勢】— 13 週實績與季末衝刺（折線圖）

> 🎯 **思維重點**：「時間 ＋ 指標」看變化，用【折線圖】；實績與目標單位相同（元），可以畫在同一個 Y 軸。季末兩週的高峰要標出並說明。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請分析全公司 13 週的實績變化：
1. 依週次加總全公司的週實績與週目標，並計算每週達成率。
2. 畫折線圖：X 軸為週次（第 1～13 週），兩條線分別為週實績與週目標（單位：萬元），Y 軸從 0 開始，加圖例。
3. 用不同底色標示第 12、13 週（季末衝刺），並在圖上註明。
4. 用文字說明：季末兩週的達成率是多少？能不能因此說「業績正在成長」？
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
weekly = df.groupby('週次')[['週目標', '週實績']].sum() / 10000
weekly['達成率(%)'] = (weekly['週實績'] / weekly['週目標'] * 100).round(1)
display(weekly.round(1).T)

plt.figure(figsize=(10, 5))
plt.plot(weekly.index, weekly['週實績'], marker='o', color='#3498db', linewidth=2, label='週實績')
plt.plot(weekly.index, weekly['週目標'], color='gray', linestyle='--', label='週目標')
plt.axvspan(11.5, 13.5, color='#fdebd0', label='季末衝刺（第 12～13 週）')
plt.xticks(weekly.index, [f'第{w}週' for w in weekly.index], rotation=45)
plt.ylim(0, weekly['週實績'].max() * 1.15)
plt.title('全公司每週實績與目標（2025 第一季）', fontsize=13, fontweight='bold')
plt.ylabel('金額（萬元）')
plt.legend(loc='lower left')
plt.show()
print(f"💡 第 12、13 週達成率為 {weekly.loc[12, '達成率(%)']}%、{weekly.loc[13, '達成率(%)']}%，明顯高於其他週。")
print("⚠️ 邊界提醒：這是季末衝業績的現象，不能說「業績正在成長」，也不能預測下一季。")

---

## 🎯 學習驗收與成果回顧

完成目前選擇的層次後，請檢視你是否能親自驗收：
- [ ] **必做**：整季達成率北區 116.0% 最高；能說明它和「平均週達成率」為什麼不同。
- [ ] **報表應用**：Excel 有千分位與百分比格式，未達標標淺紅底，數值可再計算。
- [ ] **選做挑戰**：找出未達標業務並同時看金額；季末衝刺只描述現象，不預測。
- [ ] **AI 協作**：能說明自己定義的比率、AI 幫忙執行的事，以及自己如何確認結果。